# Data Visualisation with Python and AI Tools

## Code first, prompt second, think throughout

**Workshop dataset:** TMDB Movies dataset  
**Main idea:** Python builds the chart. AI supports the learning. Humans decide what is meaningful, clear, and responsible.

### What you will learn
By the end of this workshop, you should be able to:

1. Choose suitable visualisation types for different data questions.
2. Create common visualisations using Python libraries such as `matplotlib`, `seaborn`, and `plotly`.
3. Use AI tools to explain, recreate, improve, and extend Python visualisation code.
4. Interpret visual patterns critically and avoid overclaiming.

### Today’s learning loop
For each visualisation, we will use this pattern:

1. **Ask a data question**
2. **Study and run the Python script first**
3. **Interpret the chart**
4. **Use an AI prompt to recreate, explain, or improve the chart**
5. **Compare Python and AI outputs**
6. **Reflect on what we learned**


# 0. Setup: Import Libraries

Run the cell below first. It imports the Python libraries used in this workshop.


In [4]:
# Core data libraries
import pandas as pd
import numpy as np
import json

# Static visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Interactive visualisation library
import plotly.express as px

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print('✅ Libraries imported successfully!')


✅ Libraries imported successfully!


# 1. Load the Dataset

We will use the **TMDB Movies dataset**. You can choose either option:

- **Option A:** Upload `tmdb_5000_movies.csv` from your computer.
- **Option B:** Load it from Kaggle server if your Colab environment supports `kagglehub`.

Run only one option.


## Option A: Upload from your computer


In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'✅ Dataset loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


## Option B: Load from Kaggle server

If this option does not work, use Option A instead.


In [15]:
try:
    import kagglehub
    import pandas as pd
    import os

    path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

    movies_file = os.path.join(path, "tmdb_5000_movies.csv")
    credits_file = os.path.join(path, "tmdb_5000_credits.csv")

    movies = pd.read_csv(movies_file, encoding="cp1252")
    credits = pd.read_csv(credits_file, encoding="cp1252")

    print(f"✅ Movies dataset loaded! Shape: {movies.shape[0]} rows, {movies.shape[1]} columns")
    print(f"✅ Credits dataset loaded! Shape: {credits.shape[0]} rows, {credits.shape[1]} columns")

    display(movies.head())

except Exception as e:
    print("⚠️ Loading did not work in this environment.")
    print(e)

⚠️ Loading did not work in this environment.
'charmap' codec can't decode byte 0x81 in position 74135: character maps to <undefined>


In [13]:
# Optional: Load from Kaggle server
# If this cell fails, use Option A above.

try:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    file_path = 'tmdb_5000_movies.csv'

    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        'tmdb/tmdb-movie-metadata',
        file_path
    )

    print(f'✅ Dataset loaded from Kaggle! Shape: {df.shape[0]} rows, {df.shape[1]} columns')
    display(df.head())

except Exception as e:
    print('⚠️ Kaggle loading did not work in this environment. Please use Option A.')
    print(e)


100%|██████████| 1.58M/1.58M [00:00<00:00, 1.76MB/s]

⚠️ Kaggle loading did not work in this environment. Please use Option A.
Error reading file: 'utf-8' codec can't decode byte 0x94 in position 11: invalid start byte


# 2. Quick Data Preparation

We will keep data preparation short because this is a visualisation workshop.  
We only create the variables needed for visualisation:

- `primary_genre`: first genre extracted from the JSON-like `genres` column
- `release_year`: year extracted from `release_date`
- Clean zero values in `budget`, `revenue`, and `runtime`


In [ ]:
# Check basic structure
print('Dataset shape:', df.shape)
display(df.head())
print('\nColumn names:')
print(df.columns.tolist())


In [ ]:
# Extract the first genre from the JSON-like genres column

def extract_primary_genre(genre_json):
    try:
        genres = json.loads(genre_json)
        if genres and len(genres) > 0:
            return genres[0]['name']
        else:
            return 'Unknown'
    except:
        return 'Unknown'

if 'genres' in df.columns:
    df['primary_genre'] = df['genres'].apply(extract_primary_genre)
else:
    print('⚠️ genres column not found')

# Extract release year
if 'release_date' in df.columns:
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df['release_year'] = df['release_date'].dt.year
else:
    print('⚠️ release_date column not found')

# Replace zeros with NaN for selected columns, because zeros may represent missing information
columns_to_clean = ['budget', 'revenue', 'runtime']
for col in columns_to_clean:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)

# Keep movies with sufficient votes for more meaningful visualisation
if 'vote_count' in df.columns and 'vote_average' in df.columns:
    df = df[(df['vote_average'].notna()) & (df['vote_count'] > 50)].copy()

print('✅ Data preparation complete!')
print(f'Current dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df[['title', 'primary_genre', 'release_year', 'vote_average', 'vote_count', 'budget', 'revenue']].head())


# 3. Introduce the Visualisation Mindset

Before we plot, ask: **What are we trying to see?**

| If we want to see... | We are asking... | Suitable visualisation |
|---|---|---|
| Shape or spread | How is one variable distributed? | Histogram, density plot, box plot |
| Ranking or difference | Which group is higher or lower? | Bar chart, box plot |
| Relationship | Do two variables move together? | Scatter plot, bubble chart |
| Time pattern | How does something change over time? | Line chart |
| Many relationships | Which variables are connected? | Heatmap |
| Concentration | Where do many points gather? | 2D density / hexbin plot |
| Multiple dimensions | How do 3 or more variables interact? | Bubble chart, 3D plot |
| Exploration | Can I hover, zoom, or identify details? | Plotly interactive chart |

## Mini activity: Chart Match Challenge
Match each question with a suitable chart type:

1. What is the distribution of movie ratings?
2. Which genre has the highest average rating?
3. Do ratings vary widely across genres?
4. Is budget related to revenue?
5. Which variables are strongly related?
6. How have ratings changed over time?
7. Where are most movies concentrated by rating and vote count?
8. How do budget, revenue, and rating interact?

**Key message:** Good visualisation is not about using the most impressive chart. It is about choosing the chart that best answers the question.


# 4. AI as a Visualisation Partner

AI can help us:

- Generate Python code
- Explain unfamiliar code
- Improve chart design
- Suggest alternative chart types
- Help interpret patterns
- Warn us about limitations

But AI should not replace:

- Our question
- Our judgement
- Our understanding of the data
- Our responsibility to verify the result

## Weak prompt
```text
Analyse this data for me.
```

## Better prompt
```text
I am working with a pandas DataFrame called df.

My question is:
What is the distribution of movie ratings?

Please:
1. Suggest a suitable visualisation type.
2. Write Python code using seaborn.
3. Use the vote_average column.
4. Add a clear title and axis labels.
5. Explain what patterns I should look for.
```

**Key message:** AI responds better when our thinking is clearer.


# 5. Histogram: What Is Normal? Understanding Distribution

## Question
What is the distribution of movie ratings?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x='vote_average',
    bins=30,
    kde=True
)

plt.title('Distribution of Movie Ratings')
plt.xlabel('Movie Rating')
plt.ylabel('Number of Movies')
plt.show()


## How the Python code works

- `plt.figure(figsize=(10, 6))` controls chart size.
- `sns.histplot()` creates a histogram.
- `x='vote_average'` tells Python which column to visualise.
- `bins=30` controls how ratings are grouped.
- `kde=True` adds a smooth density curve.
- Titles and labels make the chart readable.


## Think before AI

- Where are most movie ratings concentrated?
- Are very high ratings common?
- Are there unusual low or high values?
- Does the distribution look balanced or skewed?


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a histogram in Python using seaborn to show the distribution of movie ratings.

Please recreate this chart using the vote_average column.

Requirements:
- Use seaborn
- Create a histogram
- Add a smooth density curve
- Add a clear title and axis labels
- Explain each line in beginner-friendly language
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 6. Bar Chart: Which Group Stands Out?

## Question
Which genre has the highest average rating?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
genre_ratings = (
    df.groupby('primary_genre')['vote_average']
    .mean()
    .sort_values(ascending=True)
)

plt.figure(figsize=(10, 8))

genre_ratings.plot(kind='barh')

plt.title('Average Movie Rating by Genre')
plt.xlabel('Average Rating')
plt.ylabel('Genre')
plt.show()


## How the Python code works

- `groupby('primary_genre')` groups movies by genre.
- `['vote_average'].mean()` calculates average rating for each genre.
- `sort_values(ascending=True)` sorts the ratings.
- `plot(kind='barh')` creates a horizontal bar chart.


## Think before AI

Before running the chart, predict: Which genre do you think will have the highest average rating?

After running the chart:
- Was your guess correct?
- Are the differences large or small?
- Should we trust average rating alone?


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a horizontal bar chart to compare average movie ratings by genre.

Please recreate this chart using seaborn instead of pandas plot.

Requirements:
- Group by primary_genre
- Calculate average vote_average
- Sort genres from highest to lowest
- Create a horizontal bar chart
- Add clear title and labels
- Add value labels at the end of each bar
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 7. Box Plot: Averages Can Hide Stories

## Question
Do movie ratings vary widely across genres?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
plt.figure(figsize=(14, 8))

sns.boxplot(
    data=df,
    x='primary_genre',
    y='vote_average'
)

plt.title('Distribution of Movie Ratings by Genre')
plt.xlabel('Genre')
plt.ylabel('Movie Rating')
plt.xticks(rotation=45, ha='right')
plt.show()


## How the Python code works

- A box plot shows median, spread, and outliers.
- `x='primary_genre'` places genres on the x-axis.
- `y='vote_average'` shows the rating distribution.
- `plt.xticks(rotation=45)` makes genre labels readable.


## Think before AI

- Which genre has a wide spread of ratings?
- Which genre seems more consistent?
- Are there many outliers?
- Why is this more informative than average rating alone?


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a box plot showing vote_average by primary_genre.

Please explain:
1. What the box plot shows.
2. How to read the median, box, whiskers, and outliers.
3. What students should look for when comparing genres.
4. One way to improve chart readability.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 8. Scatter Plot: Do Two Things Move Together?

## Question
Is there a relationship between movie rating and number of votes?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='vote_average',
    y='vote_count',
    alpha=0.6
)

plt.title('Relationship Between Movie Rating and Number of Votes')
plt.xlabel('Movie Rating')
plt.ylabel('Number of Votes')
plt.show()


## How the Python code works

- Each point represents one movie.
- The x-axis shows movie rating.
- The y-axis shows number of votes.
- `alpha=0.6` makes overlapping points more transparent.


## Think before AI

Use quadrant thinking:
- High rating, high votes: popular and well-rated
- High rating, low votes: possible hidden gems
- Low rating, high votes: popular but not highly rated
- Low rating, low votes: less visible movies

Remember: scatter plots can suggest relationships, but they do not prove cause and effect.


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a scatter plot with vote_average on the x-axis and vote_count on the y-axis.

Can you:
1. Recreate this plot using seaborn.
2. Add colour by primary_genre.
3. Explain what each point represents.
4. Suggest what patterns I should look for.
5. Remind me what I should not overclaim from this chart.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 9. Heatmap: Seeing Many Relationships at Once

## Question
Which numeric variables are related to each other?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
numeric_cols = [
    'budget',
    'revenue',
    'runtime',
    'popularity',
    'vote_average',
    'vote_count'
]

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap of Movie Variables')
plt.show()


## How the Python code works

- `numeric_cols` selects numeric variables.
- `df[numeric_cols].corr()` calculates correlations.
- `sns.heatmap()` visualises the correlation matrix.
- `annot=True` shows the correlation values.
- `cmap='coolwarm'` uses colour to show direction and strength.
- `fmt='.2f'` rounds values to two decimal places.


## Think before AI

- Which two variables have the strongest relationship?
- Does the relationship make sense?
- Are there any weak relationships?
- Can we say one variable causes another?

Key phrase: Correlation is a clue, not a conclusion.


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a correlation heatmap using:
budget, revenue, runtime, popularity, vote_average, and vote_count.

Please explain:
1. What a correlation heatmap shows.
2. How to interpret values close to 1, -1, and 0.
3. Which relationships may be interesting.
4. Why correlation does not mean causation.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 10. Trend Line: What Changes Over Time?

## Question
How have average movie ratings changed over time?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
yearly_ratings = (
    df.groupby('release_year')['vote_average']
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=yearly_ratings,
    x='release_year',
    y='vote_average'
)

plt.title('Average Movie Rating Over Time')
plt.xlabel('Year')
plt.ylabel('Average Rating')
plt.show()


## How the Python code works

- `groupby('release_year')` groups movies by year.
- `.mean()` calculates average rating per year.
- `.reset_index()` converts the result back into a DataFrame.
- `sns.lineplot()` shows the trend over time.


## Think before AI

- Is there an upward or downward trend?
- Are early years unstable?
- Why might some years be more volatile?
- Would a rolling average help?


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a line chart showing average movie rating over release_year.

Can you improve this chart by:
1. Adding a rolling average line.
2. Making the title and labels clearer.
3. Explaining why early years may fluctuate more.
4. Explaining what we should be careful about when interpreting the trend.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 11. Interactive Plotly Scatter: From Static Chart to Discovery Tool

## Question
Can we interact with points and identify individual movies?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
fig = px.scatter(
    df,
    x='vote_average',
    y='vote_count',
    hover_data=['title', 'primary_genre'],
    title='Interactive Scatter Plot: Rating vs Number of Votes',
    labels={
        'vote_average': 'Movie Rating',
        'vote_count': 'Number of Votes'
    }
)

fig.show()


## How the Python code works

- `plotly.express` creates interactive visualisations.
- `hover_data` controls what appears when hovering.
- `labels` improves axis names.
- `fig.show()` displays the interactive chart.


## Think before AI

Hover and find:
- One highly rated movie with many votes
- One highly rated movie with few votes
- One unusual point

Interactive charts help us move from overview to detail.


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created an interactive Plotly scatter plot.

Please modify the chart to:
1. Colour points by primary_genre.
2. Adjust point transparency.
3. Add vote_average, vote_count, title, and primary_genre to hover information.
4. Explain why interactive charts are useful for exploration.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 12. Bubble Chart: Showing More Than Two Variables

## Question
How do budget, revenue, popularity, and genre relate?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
fig = px.scatter(
    df,
    x='budget',
    y='revenue',
    size='popularity',
    color='primary_genre',
    hover_data=['title', 'vote_average'],
    log_x=True,
    log_y=True,
    title='Budget vs Revenue by Genre and Popularity',
    labels={
        'budget': 'Budget',
        'revenue': 'Revenue',
        'popularity': 'Popularity',
        'primary_genre': 'Genre'
    }
)

fig.show()


## How the Python code works

- x-axis shows budget.
- y-axis shows revenue.
- bubble size shows popularity.
- colour shows genre.
- hover data shows title and rating.
- log scale helps when budget and revenue vary widely.


## Think before AI

- Do higher-budget movies generally earn more?
- Are there low-budget movies with high revenue?
- Are there high-budget movies with low revenue?
- Is this chart informative, or too crowded?


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a Plotly bubble chart using:
- budget as x-axis
- revenue as y-axis
- popularity as bubble size
- primary_genre as colour

Can you suggest how to make this chart clearer for presentation?
Please explain which variables may make the chart too complex.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 13. 2D Density / Hexbin Plot: When Scatter Plots Get Too Crowded

## Question
Where are movies most concentrated by rating and vote count?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
plt.figure(figsize=(10, 6))

plt.hexbin(
    df['vote_average'],
    df['vote_count'],
    gridsize=30,
    cmap='Blues'
)

plt.colorbar(label='Number of Movies')
plt.title('Concentration of Movies by Rating and Vote Count')
plt.xlabel('Movie Rating')
plt.ylabel('Number of Votes')
plt.show()


## How the Python code works

- `hexbin()` groups many points into hexagonal bins.
- Darker areas represent more movies.
- This is useful when scatter plots become overcrowded.


## Think before AI

- What does this chart show that the scatter plot did not show clearly?
- Where is the densest area?
- Are outliers still visible?


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a hexbin plot to show concentration of movies by rating and vote count.

Please explain:
1. Why a hexbin plot can be better than a scatter plot when there are many points.
2. How to interpret darker and lighter areas.
3. How I can improve the title, labels, or colour scale.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 14. 3D Scatter Plot: Impressive, But Is It Useful?

## Question
How do budget, revenue, and rating relate in three dimensions?

## Python first
Run the Python code below first. Try to understand the logic before asking AI.


In [ ]:
fig = px.scatter_3d(
    df,
    x='budget',
    y='revenue',
    z='vote_average',
    color='primary_genre',
    hover_data=['title'],
    title='3D View of Budget, Revenue, and Rating'
)

fig.show()


## How the Python code works

- x-axis shows budget.
- y-axis shows revenue.
- z-axis shows rating.
- colour shows genre.
- interactive rotation helps exploration.


## Think before AI

- Is this chart visually exciting?
- Is this chart easy to understand?
- Would you use this in a report?
- Could a simpler 2D chart communicate better?

Key message: Advanced charts are not always better charts.


## AI prompt after Python
Copy this prompt into Gemini / ChatGPT / Claude **after** you understand the Python version.

```text
I created a 3D scatter plot using budget, revenue, and vote_average.

Please explain:
1. One benefit of using a 3D chart.
2. One limitation of using a 3D chart.
3. When a simpler 2D chart may be better.
4. How I can present this chart responsibly.
```

## Compare and reflect
- Did AI generate similar code?
- Did AI use a different library or function?
- Which version is easier to understand?
- Which version is better for presentation?
- Did AI add anything useful, or did it make assumptions?


# 15. Student Challenge: Can AI Recreate What Python Taught Us?

Choose one visualisation from the workshop:

- Histogram
- Bar chart
- Box plot
- Scatter plot
- Heatmap
- Trend line
- Interactive scatter
- Bubble chart
- Hexbin plot
- 3D scatter plot

## Your task

1. Start from the Python code shown in class.
2. Run it and understand the result.
3. Ask AI to recreate a similar visualisation.
4. Compare your original Python code with AI-generated code.
5. Improve one part of the chart.
6. Write one interpretation and one caution.

## Sharing format

```text
Chart type:
My question:
What Python helped me understand:
What AI helped me improve:
One insight:
One caution:
```


In [ ]:
# ═══════════════════════════════════════════════════
# Student Challenge Workspace
# Paste or write your Python visualisation code here.
# ═══════════════════════════════════════════════════


# 16. Wrap-up: What Makes a Good Data Visualisation?

Before making a chart, ask:

1. What question am I answering?
2. Who is my audience?
3. What chart best fits the question?
4. What pattern do I see?
5. What alternative explanation might exist?
6. What should I not overclaim?

## Final message

**Python builds the chart.**  
**AI supports the learning.**  
**Humans decide what is meaningful, clear, and responsible.**
